In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [6]:
%pip install mlflow dagshub xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 74.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import dagshub
import mlflow
dagshub.init(repo_owner='amama22', repo_name='ieee-cis-fraud-detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=67ee2777-33cd-483d-9095-b70ae5cb80dd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6094a3b389b06db0fda821c14d1ee034fbee8523961ab29f8f51d9d94440dfbd




Accessing as amama22

Initialized MLflow to track repo "amama22/ieee-cis-fraud-detection"

Repository amama22/ieee-cis-fraud-detection initialized!

In [8]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

print("Train transaction shape:", train.shape)
print("Train identity shape:", train_id.shape)
print("Test transaction shape:", test.shape)
print("Test identity shape:", test_id.shape)

Train transaction shape: (590540, 394)
Train identity shape: (144233, 41)
Test transaction shape: (506691, 393)
Test identity shape: (141907, 41)


In [7]:
print(train.head())
print("\nColumns:", train.columns.tolist())

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  0.0   0.0   0.0   0.0   0.0  0.0   

  V336  V337  V338  V339  
0  NaN   NaN   NaN   NaN  
1  NaN   NaN   NaN  

In [8]:
null_pct = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
print(null_pct.head(40))

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D9       87.312290
D8       87.312290
V153     86.123717
V149     86.123717
V141     86.123717
V146     86.123717
V154     86.123717
V162     86.123717
V142     86.123717
V158     86.123717
V161     86.123717
V157     86.123717
V138     86.123717
V139     86.123717
V148     86.123717
V140     86.123717
V155     86.123717
V156     86.123717
V147     86.123717
V163     86.123717
V143     86.122701
V145     86.122701
V144     86.122701
V165     86.122701
V164     86.122701
V152     86.122701
V150     86.122701
V151     86.122701
V160     86.122701
V159     86.122701
V166     86.122701
V329     86.054967
V328     86.054967
V326     86.054967
dtype: float64


In [9]:
print(train['isFraud'].value_counts())
print(train['isFraud'].value_counts(normalize=True))

isFraud
0    569877
1     20663
Name: count, dtype: int64
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [10]:
merged = train.merge(train_id, on='TransactionID', how='left')
has_identity = merged['id_01'].notna().sum()
print(f"Transactions WITH identity: {has_identity} ({has_identity/len(train)*100:.1f}%)")
print(f"Transactions WITHOUT identity: {len(train) - has_identity}")

Transactions WITH identity: 144233 (24.4%)
Transactions WITHOUT identity: 446307


In [11]:
null_pct = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)

print("Columns with >80% null:", (null_pct > 80).sum())
print("Columns with 50-80% null:", ((null_pct > 50) & (null_pct <= 80)).sum())
print("Columns with 20-50% null:", ((null_pct > 20) & (null_pct <= 50)).sum())
print("Columns with 0-20% null:", ((null_pct > 0) & (null_pct <= 20)).sum())
print("Columns with 0% null:", (null_pct == 0).sum())

Columns with >80% null: 55
Columns with 50-80% null: 119
Columns with 20-50% null: 38
Columns with 0-20% null: 162
Columns with 0% null: 20


In [12]:
cols = train.columns.tolist()
groups = {
    'V_cols': [c for c in cols if c.startswith('V')],
    'C_cols': [c for c in cols if c.startswith('C')],
    'D_cols': [c for c in cols if c.startswith('D')],
    'M_cols': [c for c in cols if c.startswith('M')],
    'card_cols': [c for c in cols if c.startswith('card')],
    'addr_cols': [c for c in cols if c.startswith('addr')],
}
for g, cols_list in groups.items():
    print(f"{g}: {len(cols_list)} columns")

V_cols: 339 columns
C_cols: 14 columns
D_cols: 15 columns
M_cols: 9 columns
card_cols: 6 columns
addr_cols: 2 columns


In [13]:
print(train.dtypes.value_counts())
print("\nObject columns:", train.select_dtypes('object').columns.tolist())

float64    376
object      14
int64        4
Name: count, dtype: int64

Object columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']


In [14]:
for col in ['M1','M2','M3','M4','M5','M6','M7','M8','M9']:
    print(f"{col}: {train[col].value_counts().to_dict()} | null: {train[col].isna().sum()}")

M1: {'T': 319415, 'F': 25} | null: 271100
M2: {'T': 285468, 'F': 33972} | null: 271100
M3: {'T': 251731, 'F': 67709} | null: 271100
M4: {'M0': 196405, 'M2': 59865, 'M1': 52826} | null: 281444
M5: {'F': 132491, 'T': 107567} | null: 350482
M6: {'F': 227856, 'T': 193324} | null: 169360
M7: {'F': 211374, 'T': 32901} | null: 346265
M8: {'F': 155251, 'T': 89037} | null: 346252
M9: {'T': 205656, 'F': 38632} | null: 346252


In [15]:
print("P_emaildomain unique:", train['P_emaildomain'].nunique())
print("R_emaildomain unique:", train['R_emaildomain'].nunique())
print("\nTop 10 P_emaildomain:\n", train['P_emaildomain'].value_counts().head(10))

P_emaildomain unique: 59
R_emaildomain unique: 60

Top 10 P_emaildomain:
 P_emaildomain
gmail.com        228355
yahoo.com        100934
hotmail.com       45250
anonymous.com     36998
aol.com           28289
comcast.net        7888
icloud.com         6267
outlook.com        5096
msn.com            4092
att.net            4033
Name: count, dtype: int64


In [16]:
print(train_id.dtypes.value_counts())
print("\nObject cols in identity:", train_id.select_dtypes('object').columns.tolist())
null_id = (train_id.isnull().sum() / len(train_id) * 100).sort_values(ascending=False)
print("\nIdentity null % (top 20):\n", null_id.head(20))

float64    23
object     17
int64       1
Name: count, dtype: int64

Object cols in identity: ['id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

Identity null % (top 20):
 id_24         96.708798
id_25         96.441868
id_07         96.425922
id_08         96.425922
id_21         96.423149
id_26         96.420375
id_23         96.416215
id_27         96.416215
id_22         96.416215
id_18         68.722137
id_04         54.016071
id_03         54.016071
id_33         49.187079
id_10         48.052110
id_09         48.052110
id_30         46.222432
id_32         46.207872
id_34         46.056034
id_14         44.503685
DeviceInfo    17.726179
dtype: float64


In [17]:
print(train['TransactionAmt'].describe())
print("\ncard4 values:", train['card4'].value_counts().to_dict())
print("card6 values:", train['card6'].value_counts().to_dict())
print("ProductCD values:", train['ProductCD'].value_counts().to_dict())

count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

card4 values: {'visa': 384767, 'mastercard': 189217, 'american express': 8328, 'discover': 6651}
card6 values: {'debit': 439938, 'credit': 148986, 'debit or credit': 30, 'charge card': 15}
ProductCD values: {'W': 439670, 'C': 68519, 'R': 37699, 'H': 33024, 'S': 11628}


In [18]:
v_null = (train[[c for c in train.columns if c.startswith('V')]].isnull().sum() / len(train) * 100)
print("V col null % distribution:")
print(v_null.value_counts().sort_index())

V col null % distribution:
0.002032     32
0.053172     43
0.214888     11
12.881939    23
13.055170    22
15.098723    20
28.612626    18
47.293494    11
76.053104    16
76.323534    19
76.355370    31
77.913435    46
86.054967    18
86.122701    11
86.123717    18
Name: count, dtype: int64


In [19]:
print(train.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False))

ProductCD
C    0.116873
S    0.058996
H    0.047662
R    0.037826
W    0.020399
Name: isFraud, dtype: float64


In [20]:
fraud_by_email = train.groupby('P_emaildomain')['isFraud'].mean().sort_values(ascending=False)
print(fraud_by_email.head(10))

P_emaildomain
protonmail.com    0.407895
mail.com          0.189624
outlook.es        0.130137
aim.com           0.126984
outlook.com       0.094584
hotmail.es        0.065574
live.com.mx       0.054740
hotmail.com       0.052950
gmail.com         0.043542
yahoo.fr          0.034965
Name: isFraud, dtype: float64


In [21]:
print("Skewness raw:", train['TransactionAmt'].skew())
print("Skewness log:", np.log1p(train['TransactionAmt']).skew())

Skewness raw: 14.374489573829827
Skewness log: 0.4882909971465291


In [9]:
train = train.merge(train_id, on='TransactionID', how='left')
test = test.merge(test_id, on='TransactionID', how='left')

TARGET = 'isFraud'
X = train.drop(columns=[TARGET, 'TransactionID'])
y = train[TARGET]
X_test = test.drop(columns=['TransactionID'])

print("Train shape after merge:", X.shape)
print("Test shape after merge:", X_test.shape)

Train shape after merge: (590540, 432)
Test shape after merge: (506691, 432)


In [23]:
null_pct = (X.isnull().sum() / len(X) * 100)

v1_drop = null_pct[null_pct > 80].index.tolist()
v2_drop = null_pct[null_pct > 50].index.tolist()
v3_drop = null_pct[null_pct > 90].index.tolist()

print(f"Variation 1 (>80% null) - columns to drop: {len(v1_drop)}")
print(f"Variation 2 (>50% null) - columns to drop: {len(v2_drop)}")
print(f"Variation 3 (>90% null) - columns to drop: {len(v3_drop)}")
print(f"\nTotal columns in X: {X.shape[1]}")
print(f"V1 remaining: {X.shape[1] - len(v1_drop)}")
print(f"V2 remaining: {X.shape[1] - len(v2_drop)}")
print(f"V3 remaining: {X.shape[1] - len(v3_drop)}")

Variation 1 (>80% null) - columns to drop: 74
Variation 2 (>50% null) - columns to drop: 214
Variation 3 (>90% null) - columns to drop: 12

Total columns in X: 432
V1 remaining: 358
V2 remaining: 218
V3 remaining: 420


In [24]:
print("M1 value counts:\n", X['M1'].value_counts(dropna=False))
print("\nM1 unique non-null values:", X['M1'].dropna().nunique())

M1 value counts:
 M1
T      319415
NaN    271100
F          25
Name: count, dtype: int64

M1 unique non-null values: 2


In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

# cleaning

In [17]:
class CleaningTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, null_threshold=0.8):
        self.null_threshold = null_threshold
    
    def fit(self, X, y=None):
        null_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = null_pct[null_pct > self.null_threshold].index.tolist()
        
        self.always_drop_ = [c for c in ['M1'] if c in X.columns]
        self.cols_to_drop_ = list(set(self.high_null_cols_ + self.always_drop_))
        
        if 'card6' in X.columns:
            card6_counts = X['card6'].value_counts()
            self.rare_card6_ = card6_counts[card6_counts < 100].index.tolist()
        else:
            self.rare_card6_ = []
            
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])
        if 'card6' in X.columns:
            X['card6'] = X['card6'].apply(
                lambda x: 'other' if x in self.rare_card6_ else x
            )
        return X

In [8]:
mlflow.set_experiment("XGBoost_Training")

<Experiment: artifact_location='mlflow-artifacts:/4e297b576b684751992c5ac2a3428e7b', creation_time=1777655077469, experiment_id='0', last_update_time=1777655077469, lifecycle_stage='active', name='XGBoost_Training', tags={}, trace_location=None, workspace='default'>

In [33]:
# drop >80% null
with mlflow.start_run(run_name="XGBoost_Cleaning_V1"):
    cleaner_v1 = CleaningTransformer(null_threshold=0.80)
    cleaner_v1.fit(X)
    X_v1 = cleaner_v1.transform(X)
    
    mlflow.log_param("null_threshold", 0.80)
    mlflow.log_param("always_dropped", "M1")
    mlflow.log_param("card6_rare_grouped", cleaner_v1.rare_card6_)
    mlflow.log_metric("cols_dropped", len(cleaner_v1.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v1.shape[1])
    mlflow.log_metric("rows", X_v1.shape[0])
    
    print(f"V1: {X.shape[1]} → {X_v1.shape[1]} columns")
    print(f"Dropped: {cleaner_v1.cols_to_drop_[:5]}... (total {len(cleaner_v1.cols_to_drop_)})")

# drop >50% null
with mlflow.start_run(run_name="XGBoost_Cleaning_V2"):
    cleaner_v2 = CleaningTransformer(null_threshold=0.50)
    cleaner_v2.fit(X)
    X_v2 = cleaner_v2.transform(X)
    
    mlflow.log_param("null_threshold", 0.50)
    mlflow.log_param("always_dropped", "M1")
    mlflow.log_param("card6_rare_grouped", cleaner_v2.rare_card6_)
    mlflow.log_metric("cols_dropped", len(cleaner_v2.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v2.shape[1])
    mlflow.log_metric("rows", X_v2.shape[0])
    
    print(f"V2: {X.shape[1]} → {X_v2.shape[1]} columns")

# drop >90% null
with mlflow.start_run(run_name="XGBoost_Cleaning_V3"):
    cleaner_v3 = CleaningTransformer(null_threshold=0.90)
    cleaner_v3.fit(X)
    X_v3 = cleaner_v3.transform(X)
    
    mlflow.log_param("null_threshold", 0.90)
    mlflow.log_param("always_dropped", "M1")
    mlflow.log_param("card6_rare_grouped", cleaner_v3.rare_card6_)
    mlflow.log_metric("cols_dropped", len(cleaner_v3.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v3.shape[1])
    mlflow.log_metric("rows", X_v3.shape[0])
    
    print(f"V3: {X.shape[1]} → {X_v3.shape[1]} columns")

V1: 432 → 357 columns
Dropped: ['id_14', 'id_24', 'V140', 'V154', 'D14']... (total 75)
🏃 View run XGBoost_Cleaning_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/00ed382bbace457b95ede4df20283dc6
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V2: 432 → 217 columns
🏃 View run XGBoost_Cleaning_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/b32e1fa3e3b7445182895144c21be388
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V3: 432 → 419 columns
🏃 View run XGBoost_Cleaning_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/25a2764f89b747f5bd4707de292b6045
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0


# Feature Engineering

In [12]:
class FeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, version=1):
        self.version = version
    
    def fit(self, X, y=None):
        if self.version >= 2 and 'P_emaildomain' in X.columns:
            self.email_freq_ = X['P_emaildomain'].value_counts(normalize=True).to_dict()
        
        if self.version == 3 and 'card1' in X.columns:
            self.card1_amt_mean_ = X.groupby('card1')['TransactionAmt'].mean().to_dict()
            self.global_amt_mean_ = X['TransactionAmt'].mean()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        if 'TransactionAmt' in X.columns:
            X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        
        if 'id_01' in X.columns:
            X['has_identity'] = X['id_01'].notna().astype(int)
        
        if self.version >= 2 and 'P_emaildomain' in X.columns:
            high_risk = ['protonmail.com', 'mail.com', 'outlook.es', 'aim.com']
            mid_risk  = ['outlook.com', 'hotmail.es', 'live.com.mx', 'hotmail.com']
            X['email_risk_tier'] = X['P_emaildomain'].apply(
                lambda x: 2 if x in high_risk else (1 if x in mid_risk else 0)
            )
            X['P_emaildomain_freq'] = (X['P_emaildomain']
                                       .map(self.email_freq_)
                                       .fillna(0))
        
        if self.version == 3:
            if 'card4' in X.columns and 'card6' in X.columns:
                X['card_type'] = (X['card4'].astype(str) 
                                  + '_' + X['card6'].astype(str))
            if 'card1' in X.columns and 'TransactionAmt' in X.columns:
                X['amt_vs_card_mean'] = (
                    X['TransactionAmt'] / 
                    X['card1'].map(self.card1_amt_mean_)
                    .fillna(self.global_amt_mean_)
                )
        
        return X

In [35]:
with mlflow.start_run(run_name="XGBoost_Feature_Engineering_V1"):
    fe_v1 = FeatureEngineer(version=1)
    fe_v1.fit(X_v1)
    X_fe_v1 = fe_v1.transform(X_v1)
    
    new_features_v1 = ['log_TransactionAmt', 'has_identity']
    mlflow.log_param("version", 1)
    mlflow.log_param("new_features", new_features_v1)
    mlflow.log_metric("features_after_engineering", X_fe_v1.shape[1])
    print(f"V1: {X_v1.shape[1]} → {X_fe_v1.shape[1]} columns")
    print(f"New features: {new_features_v1}")

with mlflow.start_run(run_name="XGBoost_Feature_Engineering_V2"):
    fe_v2 = FeatureEngineer(version=2)
    fe_v2.fit(X_v2)
    X_fe_v2 = fe_v2.transform(X_v2)
    
    new_features_v2 = ['log_TransactionAmt', 'has_identity', 
                       'email_risk_tier', 'P_emaildomain_freq']
    mlflow.log_param("version", 2)
    mlflow.log_param("new_features", new_features_v2)
    mlflow.log_metric("features_after_engineering", X_fe_v2.shape[1])
    print(f"V2: {X_v2.shape[1]} → {X_fe_v2.shape[1]} columns")
    print(f"New features: {new_features_v2}")

with mlflow.start_run(run_name="XGBoost_Feature_Engineering_V3"):
    fe_v3 = FeatureEngineer(version=3)
    fe_v3.fit(X_v3)
    X_fe_v3 = fe_v3.transform(X_v3)
    
    new_features_v3 = ['log_TransactionAmt', 'has_identity', 'email_risk_tier',
                       'P_emaildomain_freq', 'card_type', 'amt_vs_card_mean']
    mlflow.log_param("version", 3)
    mlflow.log_param("new_features", new_features_v3)
    mlflow.log_metric("features_after_engineering", X_fe_v3.shape[1])
    print(f"V3: {X_v3.shape[1]} → {X_fe_v3.shape[1]} columns")
    print(f"New features: {new_features_v3}")

V1: 357 → 359 columns
New features: ['log_TransactionAmt', 'has_identity']
🏃 View run XGBoost_Feature_Engineering_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/3551a233c6884cbe8923a73fc2170cc1
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V2: 217 → 220 columns
New features: ['log_TransactionAmt', 'has_identity', 'email_risk_tier', 'P_emaildomain_freq']
🏃 View run XGBoost_Feature_Engineering_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/70a1d2572f8d4dba819cbb0b86e85ee5
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V3: 419 → 425 columns
New features: ['log_TransactionAmt', 'has_identity', 'email_risk_tier', 'P_emaildomain_freq', 'card_type', 'amt_vs_card_mean']
🏃 View run XGBoost_Feature_Engineering_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/f344d70282034f328

# feature selection

In [13]:
from sklearn.preprocessing import OrdinalEncoder
class CategoricalEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        self.m_cols_ = [c for c in X.columns 
                        if c.startswith('M') and X[c].dtype == 'object']
        self.other_cat_cols_ = [c for c in X.columns 
                                if X[c].dtype == 'object' 
                                and c not in self.m_cols_]
        
        self.ordinal_enc_ = OrdinalEncoder(
            handle_unknown='use_encoded_value', 
            unknown_value=-1
        )
        if self.other_cat_cols_:
            self.ordinal_enc_.fit(X[self.other_cat_cols_].astype(str))
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.m_cols_:
            if col in X.columns:
                X[col] = X[col].map({'T': 1, 'F': 0}).fillna(-1)
        
        if self.other_cat_cols_:
            existing = [c for c in self.other_cat_cols_ if c in X.columns]
            X[existing] = self.ordinal_enc_.transform(X[existing].astype(str))
        
        return X

In [24]:
enc_v1 = CategoricalEncoder()
enc_v1.fit(X_fe_v1)
X_enc_v1 = enc_v1.transform(X_fe_v1)

enc_v2 = CategoricalEncoder()
enc_v2.fit(X_fe_v2)
X_enc_v2 = enc_v2.transform(X_fe_v2)

enc_v3 = CategoricalEncoder()
enc_v3.fit(X_fe_v3)
X_enc_v3 = enc_v3.transform(X_fe_v3)

print("Any remaining object columns in V1?", 
      X_enc_v1.select_dtypes('object').columns.tolist())
print("Any remaining object columns in V2?", 
      X_enc_v2.select_dtypes('object').columns.tolist())
print("Any remaining object columns in V3?", 
      X_enc_v3.select_dtypes('object').columns.tolist())
print(f"\nShapes: V1={X_enc_v1.shape}, V2={X_enc_v2.shape}, V3={X_enc_v3.shape}")

Any remaining object columns in V1? []
Any remaining object columns in V2? []
Any remaining object columns in V3? []

Shapes: V1=(590540, 359), V2=(590540, 220), V3=(590540, 425)


In [14]:
from sklearn.feature_selection import VarianceThreshold

class VarianceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        self.selector_ = VarianceThreshold(threshold=self.threshold)
        self.selector_.fit(X)
        self.selected_cols_ = X.columns[self.selector_.get_support()].tolist()
        self.dropped_cols_ = X.columns[~self.selector_.get_support()].tolist()
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]


class CorrelationSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.dropped_cols_ = [col for col in upper.columns 
                              if any(upper[col] > self.threshold)]
        self.selected_cols_ = [c for c in X.columns 
                               if c not in self.dropped_cols_]
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]


class ImportanceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=150):
        self.top_n = top_n
    
    def fit(self, X, y=None):
        sample_size = min(50000, len(X))
        idx = np.random.choice(len(X), size=sample_size, replace=False)
        X_sample = X.iloc[idx]
        y_sample = y.iloc[idx]
        
        quick_model = XGBClassifier(
            n_estimators=50,
            max_depth=4,
            learning_rate=0.1,
            tree_method='hist',
            random_state=42,
            eval_metric='auc'
        )
        quick_model.fit(X_sample, y_sample)
        
        importances = pd.Series(
            quick_model.feature_importances_, 
            index=X.columns
        ).sort_values(ascending=False)
        
        self.selected_cols_ = importances.head(self.top_n).index.tolist()
        self.dropped_cols_ = importances.tail(len(X.columns) - self.top_n).index.tolist()
        self.importances_ = importances
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]

In [42]:
from xgboost import XGBClassifier
with mlflow.start_run(run_name="XGBoost_Feature_Selection_V1"):
    sel_v1 = VarianceSelector(threshold=0.01)
    sel_v1.fit(X_enc_v1)
    X_sel_v1 = sel_v1.transform(X_enc_v1)
    
    mlflow.log_param("method", "VarianceThreshold")
    mlflow.log_param("threshold", 0.01)
    mlflow.log_metric("features_before", X_enc_v1.shape[1])
    mlflow.log_metric("features_after", X_sel_v1.shape[1])
    mlflow.log_metric("features_dropped", len(sel_v1.dropped_cols_))
    print(f"V1 Variance: {X_enc_v1.shape[1]} → {X_sel_v1.shape[1]} columns")
    print(f"Dropped (low variance): {sel_v1.dropped_cols_}")

with mlflow.start_run(run_name="XGBoost_Feature_Selection_V2"):
    sel_v2 = CorrelationSelector(threshold=0.95)
    sel_v2.fit(X_enc_v2)
    X_sel_v2 = sel_v2.transform(X_enc_v2)
    
    mlflow.log_param("method", "CorrelationFilter")
    mlflow.log_param("threshold", 0.95)
    mlflow.log_metric("features_before", X_enc_v2.shape[1])
    mlflow.log_metric("features_after", X_sel_v2.shape[1])
    mlflow.log_metric("features_dropped", len(sel_v2.dropped_cols_))
    print(f"V2 Correlation: {X_enc_v2.shape[1]} → {X_sel_v2.shape[1]} columns")
    print(f"Dropped (high correlation): {len(sel_v2.dropped_cols_)} features")

with mlflow.start_run(run_name="XGBoost_Feature_Selection_V3"):
    sel_v3 = ImportanceSelector(top_n=150)
    sel_v3.fit(X_enc_v3, y)
    X_sel_v3 = sel_v3.transform(X_enc_v3)
    
    mlflow.log_param("method", "FeatureImportance")
    mlflow.log_param("top_n", 150)
    mlflow.log_param("sample_size_used", 50000)
    mlflow.log_metric("features_before", X_enc_v3.shape[1])
    mlflow.log_metric("features_after", X_sel_v3.shape[1])
    mlflow.log_metric("features_dropped", len(sel_v3.dropped_cols_))
    print(f"V3 Importance: {X_enc_v3.shape[1]} → {X_sel_v3.shape[1]} columns")
    print(f"Top 10 features: {sel_v3.importances_.head(10).index.tolist()}")

V1 Variance: 359 → 334 columns
Dropped (low variance): ['M4', 'V1', 'V14', 'V27', 'V28', 'V41', 'V65', 'V68', 'V88', 'V89', 'V107', 'V108', 'V110', 'V111', 'V112', 'V113', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V240', 'V241', 'V305']
🏃 View run XGBoost_Feature_Selection_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/5ac3bced6c5c4f70be4e8b119482816b
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V2 Correlation: 220 → 155 columns
Dropped (high correlation): 65 features
🏃 View run XGBoost_Feature_Selection_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/600d73c7ef25469588db694776259f53
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
V3 Importance: 425 → 150 columns
Top 10 features: ['V258', 'V201', 'V317', 'addr2', 'C8', 'C12', 'V187', 'V91', 'V294', 'C14']
🏃 View run XGBoost_Feature_Selection_V3 at:

# training

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score

SCALE_POS_WEIGHT = (y == 0).sum() / (y == 1).sum()
print(f"scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

SKF = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_sets = {
    "underfit": {
        "n_estimators": 50,
        "max_depth": 2,
        "learning_rate": 0.3,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "auc"
    },
    "baseline": {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "auc"
    },
    "overfit": {
        "n_estimators": 1000,
        "max_depth": 12,
        "learning_rate": 0.01,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "auc",
        "reg_alpha": 0,
        "reg_lambda": 0
    },
    "tuned": {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_child_weight": 10,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "auc"
    }
}

scale_pos_weight: 27.58


In [31]:
def run_cv(X_data, y_data, params, skf):
    train_aucs, val_aucs = [], []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
        X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
        y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]
        
        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        
        train_aucs.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:,1]))
        val_aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:,1]))
    
    return np.mean(train_aucs), np.mean(val_aucs)

In [45]:
datasets = {
    "V1": X_sel_v1,
    "V2": X_sel_v2,
    "V3": X_sel_v3
}

results = {}

for var_name, X_data in datasets.items():
    results[var_name] = {}
    for param_name, params in param_sets.items():
        run_name = f"XGBoost_CV_{var_name}_{param_name}"
        print(f"Running {run_name}...")
        
        with mlflow.start_run(run_name=run_name):
            train_auc, val_auc = run_cv(X_data, y, params, SKF)
            gap = train_auc - val_auc
            
            mlflow.log_params(params)
            mlflow.log_param("variation", var_name)
            mlflow.log_param("n_features", X_data.shape[1])
            mlflow.log_metric("train_auc", train_auc)
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("overfit_gap", gap)
            
            results[var_name][param_name] = {
                "train_auc": train_auc,
                "val_auc": val_auc,
                "gap": gap
            }
            print(f"  train_auc={train_auc:.4f} | val_auc={val_auc:.4f} | gap={gap:.4f}")

Running XGBoost_CV_V1_underfit...
  train_auc=0.8869 | val_auc=0.8841 | gap=0.0028
🏃 View run XGBoost_CV_V1_underfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/e50da5c26a074552b71920027fab4fbb
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
Running XGBoost_CV_V1_baseline...
  train_auc=0.9546 | val_auc=0.9338 | gap=0.0208
🏃 View run XGBoost_CV_V1_baseline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/e287070363bb4977a1ba5ecaaee25999
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
Running XGBoost_CV_V1_overfit...
🏃 View run XGBoost_CV_V1_overfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/042676cb6ecc4875a0b13dc78b1339d7
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0


KeyboardInterrupt: 

In [29]:
param_sets["overfit"] = {
    "n_estimators": 300,
    "max_depth": 10,
    "learning_rate": 0.1,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "eval_metric": "auc",
    "reg_alpha": 0,
    "reg_lambda": 0
}

for name in param_sets:
    param_sets[name]["device"] = "cuda"

In [28]:

    cleaner_v1 = CleaningTransformer(null_threshold=0.80)
    cleaner_v1.fit(X)
    X_v1 = cleaner_v1.transform(X)
    
    cleaner_v2 = CleaningTransformer(null_threshold=0.50)
    cleaner_v2.fit(X)
    X_v2 = cleaner_v2.transform(X)

    cleaner_v3 = CleaningTransformer(null_threshold=0.90)
    cleaner_v3.fit(X)
    X_v3 = cleaner_v3.transform(X)
    
    fe_v1 = FeatureEngineer(version=1)
    fe_v1.fit(X_v1)
    X_fe_v1 = fe_v1.transform(X_v1)

    fe_v2 = FeatureEngineer(version=2)
    fe_v2.fit(X_v2)
    X_fe_v2 = fe_v2.transform(X_v2)

    fe_v3 = FeatureEngineer(version=3)
    fe_v3.fit(X_v3)
    X_fe_v3 = fe_v3.transform(X_v3)

    from xgboost import XGBClassifier
    sel_v1 = VarianceSelector(threshold=0.01)
    sel_v1.fit(X_enc_v1)
    X_sel_v1 = sel_v1.transform(X_enc_v1)

    sel_v2 = CorrelationSelector(threshold=0.95)
    sel_v2.fit(X_enc_v2)
    X_sel_v2 = sel_v2.transform(X_enc_v2)

    sel_v3 = ImportanceSelector(top_n=150)
    sel_v3.fit(X_enc_v3, y)
    X_sel_v3 = sel_v3.transform(X_enc_v3)

In [ ]:
remaining = [
    ("V1", X_sel_v1, "overfit"),
    ("V1", X_sel_v1, "tuned"),
    ("V2", X_sel_v2, "underfit"),
    ("V2", X_sel_v2, "baseline"),
    ("V2", X_sel_v2, "overfit"),
    ("V2", X_sel_v2, "tuned"),
    ("V3", X_sel_v3, "underfit"),
    ("V3", X_sel_v3, "baseline"),
    ("V3", X_sel_v3, "overfit"),
    ("V3", X_sel_v3, "tuned"),
]

for var_name, X_data, param_name in remaining:
    run_name = f"XGBoost_CV_{var_name}_{param_name}"
    print(f"Running {run_name}...")
    
    with mlflow.start_run(run_name=run_name):
        train_auc, val_auc = run_cv(X_data, y, param_sets[param_name], SKF)
        gap = train_auc - val_auc
        
        mlflow.log_params(param_sets[param_name])
        mlflow.log_param("variation", var_name)
        mlflow.log_param("n_features", X_data.shape[1])
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("overfit_gap", gap)
        
        print(f"  train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")

Running XGBoost_CV_V1_overfit...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:42:47] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:42:47] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:44:18] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:44:18] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


  train=0.9991 | val=0.9590 | gap=0.0401
🏃 View run XGBoost_CV_V1_overfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/c66c34986ad14c90ac294cd05137cce4
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
Running XGBoost_CV_V1_tuned...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:47:24] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:47:24] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:48:38] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:48:38] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


In [19]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
xgb_best_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "min_child_weight": 10,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "eval_metric": "auc"
}

xgb_best_pipeline = Pipeline([
    ('cleaner', CleaningTransformer(null_threshold=0.80)),
    ('feature_engineer', FeatureEngineer(version=1)),
    ('encoder', CategoricalEncoder()),
    ('selector', VarianceSelector(threshold=0.01)),
    ('model', XGBClassifier(**xgb_best_params))
])

xgb_best_pipeline.fit(X, y)

with mlflow.start_run(run_name="XGBoost_Best_Pipeline"):
    mlflow.log_params(xgb_best_params)
    mlflow.log_param("null_threshold", 0.80)
    mlflow.log_param("fe_version", 1)
    mlflow.log_param("selector", "VarianceThreshold_0.01")
    mlflow.log_param("variation", "V1")
    mlflow.log_metric("val_auc", 0.9473)
    mlflow.log_metric("overfit_gap", 0.0208)
    mlflow.sklearn.log_model(
        sk_model=xgb_best_pipeline,
        artifact_path="xgb_fraud_pipeline"
    )
    print("XGBoost pipeline saved to MLflow.")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:41] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:41] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/08 12:27:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 12:27:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persisten

XGBoost pipeline saved to MLflow.
🏃 View run XGBoost_Best_Pipeline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0/runs/aaeaf5b1906c432ebc9cdcb65d28eb72
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/0
